# Mistral-7B-Instruct + LlamaIndex RAG Demo

RAG pipeline using `Mistral-7B-Instruct-v0.3` loaded locally via HuggingFace and `sentence-transformers/all-mpnet-base-v2` for embeddings.

> **GPU required.** A 16GB+ GPU (A100/T4) is recommended for 8-bit quantization.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q \
    llama-index \
    llama-index-llms-huggingface \
    llama-index-embeddings-huggingface \
    transformers accelerate bitsandbytes \
    sentence-transformers pypdf

## Imports

In [ ]:
import torch
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    PromptTemplate,
)
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

## Load Data

Drop your PDF, DOCX, or TXT files into the `Data/` folder.

In [ ]:
!mkdir -p Data

documents = SimpleDirectoryReader("./Data").load_data()
print(f"Loaded {len(documents)} document(s)")

In [ ]:
documents

## Configure Prompts

Mistral Instruct uses `[INST] ... [/INST]` tags for the chat format.

In [ ]:
system_prompt = (
    "You are a Q&A assistant. Your goal is to answer questions as "
    "accurately as possible based on the instructions and context provided."
)

# Mistral Instruct prompt format
query_wrapper_prompt = PromptTemplate("<s>[INST] {query_str} [/INST]")

## Authenticate with Hugging Face Hub

Mistral models are gated — login is required.

In [ ]:
!huggingface-cli login

## Load LLM

Using [`mistralai/Mistral-7B-Instruct-v0.3`](https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3) in 8-bit to fit on a 16GB GPU.

In [ ]:
llm = HuggingFaceLLM(
    context_window=4096,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.0, "do_sample": False},
    system_prompt=system_prompt,
    query_wrapper_prompt=query_wrapper_prompt,
    tokenizer_name="mistralai/Mistral-7B-Instruct-v0.3",
    model_name="mistralai/Mistral-7B-Instruct-v0.3",
    device_map="auto",
    model_kwargs={"torch_dtype": torch.float16, "load_in_8bit": True},
)

## Configure Global Settings

`Settings` replaces the deprecated `ServiceContext`. Assignments here apply to every index and query engine created afterwards.

In [ ]:
Settings.llm          = llm
Settings.embed_model  = HuggingFaceEmbedding(model_name="sentence-transformers/all-mpnet-base-v2")
Settings.chunk_size   = 1024

## Build Vector Index

In [ ]:
index = VectorStoreIndex.from_documents(documents)

## Q&A

In [ ]:
query_engine = index.as_query_engine()

response = query_engine.query("What is correlation?")
print(response)

## Ask Your Own Question

In [ ]:
question = "Summarise the main topics covered in the documents."

response = query_engine.query(question)
print(response)